# 01 · Exploración de datos — Breast Cancer Wisconsin

Este notebook se usa **solo para el análisis exploratorio**. La carga, el preprocesamiento, el entrenamiento y la evaluación del modelo viven en `train.py`, que se ejecuta desde la línea de comandos y registra cada corrida en MLflow.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

datos = load_breast_cancer(as_frame=True)
df = datos.frame.copy()
df["diagnostico"] = df["target"].map({0: "maligno", 1: "benigno"})
print(df.shape)
df.head()

## 1. Estructura y valores faltantes

In [ ]:
df.info()
print("Valores faltantes totales:", int(df.isna().sum().sum()))

## 2. Balance de clases

In [ ]:
conteo = df["diagnostico"].value_counts()
print(conteo, "\n", (conteo / len(df)).round(3))
conteo.plot(kind="bar", color=["#4C72B0", "#DD8452"], rot=0, title="Distribución de la variable objetivo")
plt.ylabel("Casos"); plt.show()

## 3. Estadísticos descriptivos

In [ ]:
df[datos.feature_names].describe().T.round(2)

Las variables tienen escalas muy distintas (por ejemplo, `mean area` vs. `mean smoothness`), por lo que el pipeline de `train.py` incluye `StandardScaler`.

## 4. Correlación con la variable objetivo

In [ ]:
corr_obj = df[list(datos.feature_names) + ["target"]].corr()["target"].drop("target").sort_values()
corr_obj.plot(kind="barh", figsize=(7, 9), title="Correlación de cada variable con target (1 = benigno)")
plt.tight_layout(); plt.show()

## 5. Multicolinealidad entre variables

In [ ]:
plt.figure(figsize=(11, 9))
sns.heatmap(df[datos.feature_names].corr(), cmap="coolwarm", center=0, xticklabels=False)
plt.title("Matriz de correlación entre variables"); plt.show()

## 6. Distribución de las variables más discriminantes

In [ ]:
top = corr_obj.abs().sort_values(ascending=False).head(4).index.tolist()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, top):
    sns.boxplot(data=df, x="diagnostico", y=col, hue="diagnostico", ax=ax, legend=False)
    ax.set_title(col)
plt.tight_layout(); plt.show()

## Conclusiones de la exploración

- 569 observaciones, 30 variables numéricas y ningún valor faltante.
- Clases moderadamente desbalanceadas (≈ 63 % benigno / 37 % maligno), por eso la partición es **estratificada** y se reportan F1 y ROC AUC además de accuracy.
- Escalas heterogéneas, de ahí la estandarización dentro del pipeline.
- Fuerte multicolinealidad (radio, perímetro y área); un modelo de árboles como Random Forest es robusto a ello.
- Variables de tamaño y concavidad (`worst perimeter`, `worst concave points`, etc.) separan muy bien las clases, así que se esperan métricas altas.

➡️ El modelado continúa en `train.py`.